# Импорт требующихся зависимостей

In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
from urllib.parse import quote_plus

# Подоготовка необходимых переменных и функции скраппинга

In [2]:
base_url = "https://habr.com"
# Cтрока запроса!
query = "кроссплатформенные приложения на JavaScript"
query_encoded = quote_plus(query)
search_url = f"{base_url}/ru/search/?q={query_encoded}"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
}

articles = []
page_num = 1

# Скраппинг страницы
def fetch_page(url):
    response = requests.get(url, headers=headers)
    response.raise_for_status()
    return BeautifulSoup(response.text, "html.parser")


# Парсинг всех страниц поиска

Функция постранично парсит список статей и для каждой вытаскивает:

- Заголовок
- Сылку
- Месяц
- Год
- Количество просмотров
- Текстовое описание без картинок

после чего переходит на следующую страницу, пока не достигнет конца выдачи.

In [3]:
while True:
    print(f"Парсинг страницы {page_num}: {search_url}")

    soup = fetch_page(search_url)
    posts = soup.select("article.tm-articles-list__item")

    if not posts:
        print("Статьи не найдены — возможно, конец выдачи или неверный селектор.")
        break

    for item in posts:
        article_data = {}



        # Заголовок и ссылка
        title_tag = item.select_one("a.tm-title__link")
        if not title_tag:
            continue

        article_data["title"] = title_tag.text.strip()
        article_data["link"] = base_url + title_tag.get("href")


        # Месяц и год
        date_tag = item.select_one("time")
        if date_tag and date_tag.has_attr("datetime"):
            dt = date_tag["datetime"]
            year = dt[:4]
            month = dt[5:7]

            article_data["year"] = year
            article_data["month"] = month
        else:
            article_data["year"] = None
            article_data["month"] = None


        # Количество просмотров
        views_tag = item.select_one("span.tm-icon-counter__value")
        if views_tag:
            article_data["views"] = views_tag.get("title") or views_tag.text.strip()
        else:
            article_data["views"] = None


        # Текстовое описание
        snippet = item.select_one("div.tm-article-body") or item.select_one("div.article-formatted-body")
        if snippet:
            for img in snippet.select("img"):
                img.decompose()

            desc = snippet.get_text(separator=" ", strip=True)
            article_data["description"] = desc
        else:
            article_data["description"] = None




        articles.append(article_data)

    next_button = soup.select_one("a#pagination-next-page")

    if not next_button:
        print("Достигнут конец страниц.")
        break

    search_url = base_url + next_button.get("href")
    page_num += 1
    time.sleep(1)

Парсинг страницы 1: https://habr.com/ru/search/?q=%D0%BA%D1%80%D0%BE%D1%81%D1%81%D0%BF%D0%BB%D0%B0%D1%82%D1%84%D0%BE%D1%80%D0%BC%D0%B5%D0%BD%D0%BD%D1%8B%D0%B5+%D0%BF%D1%80%D0%B8%D0%BB%D0%BE%D0%B6%D0%B5%D0%BD%D0%B8%D1%8F+%D0%BD%D0%B0+JavaScript
Парсинг страницы 2: https://habr.com/ru/search/page2/?q=%D0%BA%D1%80%D0%BE%D1%81%D1%81%D0%BF%D0%BB%D0%B0%D1%82%D1%84%D0%BE%D1%80%D0%BC%D0%B5%D0%BD%D0%BD%D1%8B%D0%B5+%D0%BF%D1%80%D0%B8%D0%BB%D0%BE%D0%B6%D0%B5%D0%BD%D0%B8%D1%8F+%D0%BD%D0%B0+JavaScript
Парсинг страницы 3: https://habr.com/ru/search/page3/?q=%D0%BA%D1%80%D0%BE%D1%81%D1%81%D0%BF%D0%BB%D0%B0%D1%82%D1%84%D0%BE%D1%80%D0%BC%D0%B5%D0%BD%D0%BD%D1%8B%D0%B5+%D0%BF%D1%80%D0%B8%D0%BB%D0%BE%D0%B6%D0%B5%D0%BD%D0%B8%D1%8F+%D0%BD%D0%B0+JavaScript
Парсинг страницы 4: https://habr.com/ru/search/page4/?q=%D0%BA%D1%80%D0%BE%D1%81%D1%81%D0%BF%D0%BB%D0%B0%D1%82%D1%84%D0%BE%D1%80%D0%BC%D0%B5%D0%BD%D0%BD%D1%8B%D0%B5+%D0%BF%D1%80%D0%B8%D0%BB%D0%BE%D0%B6%D0%B5%D0%BD%D0%B8%D1%8F+%D0%BD%D0%B0+JavaScript
Парсин

# Сохранение в CSV

In [4]:
df = pd.DataFrame(articles)
df.to_csv("habr_articles.csv", index=False, encoding="utf-8")

print("\nГотово! Сохранено статей:", len(df))
print("Файл: habr_articles.csv")
df.head()


Готово! Сохранено статей: 840
Файл: habr_articles.csv


,title,link,year,month,views,description
0,Создание кроссплатформенных приложений с помощ...,https://habr.com/ru/companies/nix/articles/324...,2017,03,43170,"Предлагаем вашему вниманию перевод статьи, кот..."
1,Создаем приложение на JavaScript с помощью Rea...,https://habr.com/ru/companies/plarium/articles...,2016,06,159785,В этом уроке мы будем изучать React Native – ф...
2,"Рассказ о том, как команда фрилансеров пишет ф...",https://habr.com/ru/companies/ruvds/articles/4...,2019,06,13560,"Автор материала, перевод которого мы сегодня п..."
3,Cвежее дополнение к Visual Studio для создания...,https://habr.com/ru/companies/microsoft/articl...,2014,06,25283,
4,Разработка кроссплатформенного приложения с по...,https://habr.com/ru/articles/255653/,2015,04,83830,
